# Joint optimization of delay and SLA risk with Optuna

After the tournament, the search turns to better configurations without artificially separating preprocessing from the model. The search space includes treatment of numeric variables, feature selection, and estimator hyperparameters.

The business objective remains the same: estimate delivery delay in days and identify orders at risk of breaching the promised deadline. Optimization uses only chronological training and validation; the test period remains reserved for final evaluation.

## Reproducible and persistent search

The number of trials is deliberately configurable. Scaling (standard or robust) is searched only for linear models, ElasticNet (Zou and Hastie, 2005), Ridge (Hoerl and Kennard, 1970), and logistic regression, because the lack of a common scale across price, distance, and lead times harms their convergence; trees and boosting use no scaler. Classification uses no class weights and no `scale_pos_weight` search: imbalance is handled by isotonic calibration (Zadrozny and Elkan, 2002) fitted only on each fold's training data, the same policy as the tournament. The cell below performs a real search with Optuna (Akiba et al., 2019) when `RUN_OPTUNA` is enabled; when the notebook is used only to review the structure, it can be disabled without changing the data contract. To bound runtime, `run_optuna_study` accepts `timeout_seconds`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import optuna
from IPython.display import display

LANG = "en"
optuna.logging.set_verbosity(optuna.logging.WARNING)
RUN_OPTUNA = True
TARGET_TRIALS = 100
STUDY_VERSION = "V4_folds"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    build_order_features,
    build_modeling_frame,
    build_supervised_orders,
    read_olist_tables,
    run_optuna_study,
    expanding_window_folds,
)

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_order_features(
    orders=tables["orders"],
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)
modeling_frame = build_modeling_frame(candidate_features, supervised)
folds = expanding_window_folds(modeling_frame, timestamp_col="order_purchase_timestamp", first_val_month="2017-10", last_val_month="2018-06", embargo_days=30)
splits = {"train": folds[-1]["train"], "validation": folds[-1]["validation"]}
study_config = [
    ("regression", "ElasticNet"),
    ("regression", "Ridge"),
    ("regression", "XGBoost"),
    ("regression", "LightGBM"),
    ("regression", "CatBoost"),
    ("classification", "LogisticRegression"),
    ("classification", "XGBoost"),
    ("classification", "LightGBM"),
    ("classification", "CatBoost"),
]
storage_path = project_root / "outputs" / "optuna" / "sla_optimization.db"
optimization_rows = []
if RUN_OPTUNA:
    for task, model_name in study_config:
        print(f"[Optuna] starting {task}/{model_name} up to {TARGET_TRIALS} trials", flush=True)
        study = run_optuna_study(
            splits=splits,
            task=task,
            model_name=model_name,
            target_trials=TARGET_TRIALS,
            storage_path=storage_path,
            study_name=f"{task}_{model_name}_{STUDY_VERSION}",
            progress=True,
            folds=folds,
        )
        optimization_rows.append(
            {
                "task": task,
                "model": model_name,
                "trials": len(study.trials),
                "best_validation_score": study.best_value,
                "best_params": study.best_params,
            }
        )
optimization_results = pd.DataFrame(optimization_rows)
optimization_results["metric"] = optimization_results["task"].map(
    {"regression": "validation_rmse", "classification": "validation_average_precision"}
)
optimization_results["rank"] = 0
for task in optimization_results["task"].unique():
    mask = optimization_results["task"] == task
    optimization_results.loc[mask, "rank"] = optimization_results.loc[mask, "best_validation_score"].rank(
        method="min", ascending=task == "regression"
    )
ranked_results = optimization_results.sort_values(["task", "rank"])
display(ranked_results)


[Optuna] starting regression/ElasticNet up to 100 trials
[Optuna] regression_ElasticNet_V4_folds already has 100/100 terminal trials.
[Optuna] starting regression/Ridge up to 100 trials
[Optuna] regression_Ridge_V4_folds already has 100/100 terminal trials.
[Optuna] starting regression/XGBoost up to 100 trials
[Optuna] regression_XGBoost_V4_folds already has 100/100 terminal trials.
[Optuna] starting regression/LightGBM up to 100 trials


CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[Optuna] regression_LightGBM_V4_folds [=.......................] 1/16 trial=85 state=complete best=9.09894
[Optuna] regression_LightGBM_V4_folds [===.....................] 2/16 trial=86 state=pruned best=9.09894
[Optuna] regression_LightGBM_V4_folds [====....................] 3/16 trial=87 state=pruned best=9.09894


CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[Optuna] regression_LightGBM_V4_folds [======..................] 4/16 trial=88 state=complete best=9.09894
[Optuna] regression_LightGBM_V4_folds [=======.................] 5/16 trial=89 state=pruned best=9.09894


CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


... (362 outputs left out)

[Optuna] classification_CatBoost_V4_folds [=================.......] 72/100 trial=71 state=complete best=0.17246
[Optuna] classification_CatBoost_V4_folds [=================.......] 73/100 trial=72 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [=================.......] 74/100 trial=73 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [==================......] 75/100 trial=74 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [==================......] 76/100 trial=75 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [==================......] 77/100 trial=76 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [==================......] 78/100 trial=77 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [==================......] 79/100 trial=78 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [===================.....] 80/100 trial=79 state=pruned best=0.17246

CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
CONDA_ENV\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[Optuna] classification_CatBoost_V4_folds [======================..] 92/100 trial=91 state=complete best=0.17246
[Optuna] classification_CatBoost_V4_folds [======================..] 93/100 trial=92 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [======================..] 94/100 trial=93 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [======================..] 95/100 trial=94 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [=======================.] 96/100 trial=95 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [=======================.] 97/100 trial=96 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [=======================.] 98/100 trial=97 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [=======================.] 99/100 trial=98 state=pruned best=0.17246
[Optuna] classification_CatBoost_V4_folds [========================] 100/100 trial=99 state=pruned best=0.1724

,task,model,trials,best_validation_score,best_params,metric,rank
7,classification,LightGBM,100,0.174243,"{'n_estimators': 350, 'num_leaves': 67, 'max_d...",validation_average_precision,1
8,classification,CatBoost,100,0.172464,"{'iterations': 150, 'depth': 7, 'learning_rate...",validation_average_precision,2
6,classification,XGBoost,100,0.171975,"{'n_estimators': 450, 'max_depth': 5, 'learnin...",validation_average_precision,3
5,classification,LogisticRegression,100,0.165821,"{'C': 0.05976091728010847, 'preprocessing_scal...",validation_average_precision,4
4,regression,CatBoost,100,8.846497,"{'iterations': 200, 'depth': 9, 'learning_rate...",validation_rmse,1
2,regression,XGBoost,100,9.054772,"{'n_estimators': 200, 'max_depth': 9, 'learnin...",validation_rmse,2
3,regression,LightGBM,101,9.098937,"{'n_estimators': 400, 'num_leaves': 97, 'max_d...",validation_rmse,3
0,regression,ElasticNet,100,9.121467,"{'alpha': 0.017358062632718076, 'l1_ratio': 0....",validation_rmse,4
1,regression,Ridge,100,9.157278,"{'alpha': 70.72114131472235, 'preprocessing_sc...",validation_rmse,5


### Insight from the top-ranked models

The studies in this version (`V4_folds`) use the current protocol: validation on expanding folds, no class weights, and isotonic calibration for classification. The earlier results (`V3`) came from the old protocol and were discarded; the numbers in this section will be filled in after the notebook is re-run.

Once the table above is regenerated, models should be compared by mean fold score, and small differences treated with caution: decimal gaps are not enough to declare an operational winner. The chosen hyperparameters describe what worked best on validation; they do not prove that the attributes cause delay or breach.

This is a development ranking: the studies optimized only chronological training and validation. The next gate is to refit the leading candidates on training + validation and evaluate them once on the frozen test period, also comparing stability, complexity, and interpretability.

## Optimization stopping rule

There will be no automatic winner based only on a decimal difference. Score, validation-to-test stability, complexity, training time, and explainability will be considered. The historical dataset does not contain operational variables such as traffic, real routes, or distribution-center events; optimization can only choose better among the observed signals.

Running the notebook creates `outputs/optuna/sla_optimization.db`, a local study artifact. The final test must be run separately and documented before any production conclusion.

## References

Akiba, T., Sano, S., Yanase, T., Ohta, T., and Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining*, 2623–2631. https://doi.org/10.1145/3292500.3330701

Hoerl, A. E. and Kennard, R. W. (1970). Ridge regression: biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55–67. https://doi.org/10.1080/00401706.1970.10488634

Zadrozny, B. and Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. *Proceedings of the 8th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 694–699. https://doi.org/10.1145/775047.775151

Zou, H. and Hastie, T. (2005). Regularization and variable selection via the elastic net. *Journal of the Royal Statistical Society, Series B*, 67(2), 301–320. https://doi.org/10.1111/j.1467-9868.2005.00503.x